In [3]:
%run 00_utils.ipynb

%load_ext autoreload
%autoreload 2

Root-Verzeichnis: /home/jessh/Study/Bachelor-Thesis
Datensätze: Anti-UAV410 + CST-AntiUAV
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from pathlib import Path
import cv2, os
import numpy as np
from data_handler.dataset import AntiUAVDataset
import albumentations as A

from augmentations.ir_camara_effects import thermal_histogram_compression, thermal_histogram_shift

In [5]:
# Trainings-Frames aller Datensätze in DATASETS (Reihenfolge wie in DATASETS).
# Nur einer: AntiUAVDataset(DatasetName.CST_ANTI_UAV, split="train", debug=True)
train_data = AntiUAVDataset(DATASETS, split="train", debug=True)

# Index-Bereich je Datensatz (für IMAGE_ID / train_data[...])
_start = 0
for ds in DATASETS:
    n = sum(r.dataset is ds for r in train_data.samples)
    print(f"{ds.value:<14} Index {_start} .. {_start + n - 1}")
    _start += n

24-11-2026 17:12:42 | INFO    | [train] Anti-UAV410: 213995 frames in 200 sequences (2.5 s)
24-11-2026 17:12:44 | INFO    | [train] CST-AntiUAV: 127991 frames in 120 sequences (1.6 s)


Anti-UAV410    Index 0 .. 213994
CST-AntiUAV    Index 213995 .. 341985


In [6]:
IMAGE_ID = 100000

In [7]:
def wrap(transform):
    # Falls nur ein einzelner Transform übergeben wurde, in eine Liste packen
    if not isinstance(transform, (list, tuple)):
        transform = [transform]
        
    return A.Compose(
        transform, 
        bbox_params=A.BboxParams(format='coco', label_fields=['class_labels']),
        save_applied_params=True
    )

PROB = 1.0

In [8]:

transforms = [
    # 1. Simuliert thermisches Rauschen des Sensors
    A.GaussNoise(var_limit=(1.0, 5.0), p=PROB),
    
    # 2. Simuliert Thermal Blooming (leichtes Verschwimmen der Hitze)
    A.GaussianBlur(blur_limit=(3, 5), p=PROB),
    
    # 3. Simuliert Thermal Crossover / wechselnde Temperaturkontraste
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.2, p=PROB),
    
    # 4. Verbessert/verschlechtert lokale Kontraste (wie echte Wärmebilder)
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=PROB),
    
    # 5. Simuliert Hitzeflimmern in Bodennähe
    [A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05, p=PROB),
     A.GridDistortion(num_steps=5, distort_range=(-0.1, 0.1), p=PROB)],

    A.Downscale(
        scale_range=(0.05, 0.25),
        interpolation_pair={'downscale': cv2.INTER_NEAREST, 'upscale': cv2.INTER_LINEAR},
        p=PROB
    ),
]


transforms =[
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=PROB),

# 1. Globales Histogramm-Shifting (Simuliert globale Temperatur-Drifts)
    A.Lambda(
        image=lambda img, **kw: thermal_histogram_shift(img, delta_limit=35), 
        p=PROB
    ),
    
    # 2. Histogramm-Kompression (Simuliert Thermal Crossover / niedrigen SCR)
    A.Lambda(
        image=lambda img, **kw: thermal_histogram_compression(img, gamma_min=0.4, gamma_max=0.85), 
        p=PROB
    ),
]

b_show_org = True
for transform in transforms:
    train_data.transform = wrap(transform)
    img, target, img_org = train_data[IMAGE_ID]
    continue

    if b_show_org:
        print(f"ORG:    {target['img_path']}")
        show_img(img_org)
        print("\n")
        b_show_org = not b_show_org
        
    
    print(" ".join([k[0] for k in target["applied_transforms"]]))
    show_img(img)

/tmp/ipykernel_771388/1075926446.py:3: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(1.0, 5.0), p=PROB),
/tmp/ipykernel_771388/1075926446.py:15: UserWarning: Argument(s) 'shift_limit' are not valid for transform OpticalDistortion
  [A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05, p=PROB),
/tmp/ipykernel_771388/1075926446.py:16: UserWarning: Argument(s) 'distort_range' are not valid for transform GridDistortion
  A.GridDistortion(num_steps=5, distort_range=(-0.1, 0.1), p=PROB)],
/tmp/ipykernel_771388/1075926446.py:30: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(
/tmp/ipykernel_771388/1075926446.py:36: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(
